In [ ]:
# Run this command if pyspark is not installed or getting this error: ImportError: No module named pyspark
!pip install pyspark

In [ ]:
# importing all the python packages
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import countDistinct
from pyspark.sql.functions import col, max
from pyspark.sql.functions import expr, desc


# creating spark session to read csv files
spark = SparkSession.builder.appName("Read Parquet").getOrCreate()


In [ ]:
# provide the path to load parquet files
df = spark.read.parquet(f"part*.parquet")

In [ ]:
df.count()

500000

In [ ]:
df.printSchema()

root
 |-- HH_ID: integer (nullable = true)
 |-- CUST_ID: integer (nullable = true)
 |-- CAR_ID: integer (nullable = true)
 |-- Active HH: integer (nullable = true)
 |-- HH Start Date: string (nullable = true)
 |-- Phone Number: string (nullable = true)
 |-- ZIP : integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Referral Source: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Make: string (nullable = true)
 |-- Body Style: string (nullable = true)
 |-- Vehicle Value: double (nullable = true)
 |-- Annual Miles Driven: integer (nullable = true)
 |-- Business Use: integer (nullable = true)
 |-- Antique Vehicle: integer (nullable = true)
 |-- Lien: integer (nullable = true)
 |-- Lease: integer (nullable = true)
 |-- Driver Safety Discount: integer (nullable = true)
 |-- Vehicle Safety Discount: integer (nullable = true)
 |-- Claim Payout: integer (nullable = true)
 |-- 6 Month 

In [ ]:
hhId = df.select(countDistinct("HH_ID"))
print("The distinct count of households:")
hhId.show()

activeHHId = df.select("HH_ID", "Active HH")
activeHHId = activeHHId.withColumnRenamed("Active HH", "ActiveHH")
activeHHId = activeHHId.filter("ActiveHH == '1'")
activeHHId = activeHHId.select(countDistinct("HH_ID"))
print("The distinct count of households with Active Flag:")
activeHHId.show()

The distinct count of households:
+---------------------+
|count(DISTINCT HH_ID)|
+---------------------+
|               131353|
+---------------------+

The distinct count of households with Active Flag:
+---------------------+
|count(DISTINCT HH_ID)|
+---------------------+
|               105331|
+---------------------+



In [211]:
#df.show()
df_test = df.withColumnRenamed("Active HH", "ActiveHH")
df_withActive = df_test.filter("ActiveHH == '1'")
print("The distinct count of customers in the combined dataset:")
df_test.agg(countDistinct("CUST_ID")).show()
print("The distinct count of customers in the combined dataset with Active Flag:")
df_withActive.agg(countDistinct("CUST_ID")).show()
print("The distinct count of cars in the combined dataset:")
df_test.agg(countDistinct("CAR_ID")).show()
print("The distinct count of cars in the combined dataset with Active Flag:")
df_withActive.agg(countDistinct("CAR_ID")).show()

The distinct count of customers in the combined dataset:
+-----------------------+
|count(DISTINCT CUST_ID)|
+-----------------------+
|                 499851|
+-----------------------+

The distinct count of customers in the combined dataset with Active Flag:
+-----------------------+
|count(DISTINCT CUST_ID)|
+-----------------------+
|                 401093|
+-----------------------+

The distinct count of cars in the combined dataset:
+----------------------+
|count(DISTINCT CAR_ID)|
+----------------------+
|                372536|
+----------------------+

The distinct count of cars in the combined dataset with Active Flag:
+----------------------+
|count(DISTINCT CAR_ID)|
+----------------------+
|                315055|
+----------------------+



In [214]:
# 1. What is the average number of cars per household?

hhCarCount = (df.groupBy("HH_ID").agg(countDistinct("CAR_ID")).orderBy("HH_ID"))
hh_car_avg = hhCarCount.agg({'count(DISTINCT CAR_ID)': 'avg'})
print("The average number of cars per household:")
hh_car_avg.show()


hhCarCountActive = (df_withActive.groupBy("HH_ID").agg(countDistinct("CAR_ID")).orderBy("HH_ID"))
hh_car_avgActive = hhCarCountActive.agg({'count(DISTINCT CAR_ID)': 'avg'})
print("The average number of cars per household with Active Flag:")
hh_car_avgActive.show()


The average number of cars per household:
+---------------------------+
|avg(count(DISTINCT CAR_ID))|
+---------------------------+
|         3.8065365846231147|
+---------------------------+

The average number of cars per household with Active Flag:
+---------------------------+
|avg(count(DISTINCT CAR_ID))|
+---------------------------+
|          3.808907159335808|
+---------------------------+



In [218]:
# 2. How many cars are there by age?




print("****************************")
print(" Assumption: A particular car can belong to multiple households and customers over the period of time. So to get the count of cars by age, \n\
will be taking a record of car with the max(Age) to make sure car is counted once for an age")
print("****************************\n")

df_age = df.filter(df.DOB.isNotNull())
df_age = df_age.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))
df_age = df_age.groupBy("CAR_ID").agg({"Age":"max"})
df_age = df_age.withColumnRenamed("max(Age)", "Age")


df_age = (df_age.groupBy("Age").agg(countDistinct("CAR_ID"))).orderBy(df_age.Age)

print("The count of cars by Age:")
df_age.show()
print("****************************\n")
df_age.agg({"count(DISTINCT CAR_ID)":"sum"}).show()
print("##################################")
print("SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG")
print("##################################\n")


df_age_Active = df_withActive.filter(df.DOB.isNotNull())
df_age_Active = df_age_Active.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))
df_age_Active = df_age_Active.groupBy("CAR_ID").agg({"Age":"max"})
df_age_Active = df_age_Active.withColumnRenamed("max(Age)", "Age")


df_age_Active = (df_age_Active.groupBy("Age").agg(countDistinct("CAR_ID"))).orderBy(df_age_Active.Age)

print("The count of cars by Age:")
df_age_Active.show()
print("****************************\n")
df_age_Active.agg({"count(DISTINCT CAR_ID)":"sum"}).show()
print("****************************")



****************************
 Assumption: A particular car can belong to multiple households and customers over the period of time. So to get the count of cars by age, 
will be taking a record of car with the max(Age) to make sure car is counted once for an age
****************************

The count of cars by Age:
+---+----------------------+
|Age|count(DISTINCT CAR_ID)|
+---+----------------------+
| 15|                  4308|
| 16|                  5564|
| 17|                  5680|
| 18|                  5818|
| 19|                  5881|
| 20|                  5796|
| 21|                  6058|
| 22|                  5972|
| 23|                  6026|
| 24|                  9788|
| 25|                  6060|
| 26|                  4675|
| 27|                  4910|
| 28|                  4937|
| 29|                  4918|
| 30|                  4907|
| 31|                  5147|
| 32|                  4997|
| 33|                  5246|
| 34|                  5099|
+---+----------

In [219]:
# 3. How many cars are there by make?


print("****************************")


print(" Assumption: In the final dataset there are records where a particular car belongs to multiple State and has multiple values of Make. There is no identifier \n\
that can be used to identify a particular Make for a car. So to get the count of cars by make the logic built belong is that, \n\
will be taking a records of a particular car and combine all the various Make type in a single column with comma seperated and then will be getting the Make \n\
value from the [0] position of the new column to get a value of Make for a Car ")
print("****************************\n")

df_make = df.select("CAR_ID","Make")
df_new = df_make.groupby('CAR_ID') \
           .agg(F.concat_ws(', ', F.collect_list(df_make.Make)) \
           .alias('CAR_MAKE'))


df_make = df_new.select("CAR_ID",F.split("CAR_MAKE", ",")[0])

df_make = df_make.withColumnRenamed("split(CAR_MAKE, ,, -1)[0]","MAKE")
df_make = df_make.groupBy("MAKE").agg(countDistinct("CAR_ID"))
print("Count of Cars by Make: ")
df_make.orderBy("MAKE").show()
print("****************************\n")
df_make.agg({"count(DISTINCT CAR_ID)":"sum"}).show()
print("##################################")
print("SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG")
print("##################################\n")

df_make_Active = df_withActive.select("CAR_ID","Make")
df_new_Active = df_make_Active.groupby('CAR_ID') \
           .agg(F.concat_ws(', ', F.collect_list(df_make_Active.Make)) \
           .alias('CAR_MAKE'))


df_make_Active = df_new_Active.select("CAR_ID",F.split("CAR_MAKE", ",")[0])

df_make_Active = df_make_Active.withColumnRenamed("split(CAR_MAKE, ,, -1)[0]","MAKE")
df_make_Active = df_make_Active.groupBy("MAKE").agg(countDistinct("CAR_ID"))
print("Count of Cars by Make: ")
df_make_Active.orderBy("MAKE").show()
print("****************************\n")
df_make_Active.agg({"count(DISTINCT CAR_ID)":"sum"}).show()




****************************
 Assumption: In the final dataset there are records where a particular car belongs to multiple State and has multiple values of Make. There is no identifier 
that can be used to identify a particular Make for a car. So to get the count of cars by make the logic built belong is that, 
will be taking a records of a particular car and combine all the various Make type in a single column with comma seperated and then will be getting the Make 
value from the [0] position of the new column to get a value of Make for a Car 
****************************

Count of Cars by Make: 
+-------------+----------------------+
|         MAKE|count(DISTINCT CAR_ID)|
+-------------+----------------------+
|Manufacturer1|                 74373|
|Manufacturer2|                 74536|
|Manufacturer3|                 74558|
|Manufacturer4|                 25932|
|Manufacturer5|                 26003|
|Manufacturer6|                 48580|
|Manufacturer7|                 48554|
+---

In [199]:
# 4. Which cars are the safest?
print(" Assumption: The cars that have [Vehicle Safety Discount] = 1, [Body Style] = Truck, [Antique Vehicle] = 0 and Claim Payout == 0 are the safest")
print("****************************\n")
# Getting distinct values of Body Style
df.select("Body Style").distinct().show()
# Getting distinct values of Vehicle Safety Discount
df.select("Vehicle Safety Discount").distinct().show()
# Getting distinct values of Antique Vehicle
df.select("Antique Vehicle").distinct().show()


df_car_safe = df.filter((F.col('Body Style') == 'Truck') & (F.col('Vehicle Safety Discount') == 1)& (F.col('Antique Vehicle') == 0) & (F.col('Claim Payout') == 0))
print("The count of cars that are safest and have Body Style = Truck and Vehicle Safety Discount = 1: " + str(df_car_safe.select("CAR_ID").distinct().count()))
print("****************************\n")
print("The list of CAR_ID that are safest in the order of ascending CAR_ID :")
df_car_safe.select("CAR_ID").distinct().orderBy("CAR_ID").show()

 Assumption: The cars that have [Vehicle Safety Discount] = 1, [Body Style] = Truck, [Antique Vehicle] = 0 and Claim Payout == 0 are the safest
****************************

+----------+
|Body Style|
+----------+
|    4 door|
|       SUV|
|     Truck|
|    2 door|
+----------+

+-----------------------+
|Vehicle Safety Discount|
+-----------------------+
|                      1|
|                      0|
+-----------------------+

+---------------+
|Antique Vehicle|
+---------------+
|              1|
|              0|
+---------------+

The count of cars that are safest and have Body Style = Truck and Vehicle Safety Discount = 1: 13598
****************************

The list of CAR_ID that are safest in the order of ascending CAR_ID :
+------+
|CAR_ID|
+------+
|  1004|
|  1008|
|  1018|
|  1029|
|  1031|
|  1041|
|  1055|
|  1073|
|  1115|
|  1155|
|  1167|
|  1179|
|  1249|
|  1253|
|  1264|
|  1293|
|  1305|
|  1310|
|  1322|
|  1325|
+------+
only showing top 20 rows



In [200]:
# 5. Which cars are the most dangerous?
print(" Assumption: The cars that have [Vehicle Safety Discount] = 0, [Body Style] = 2 door, [Antique Vehicle] =1 and Claim Payout > 0 are the dangerous")
print("****************************\n")
df_car_danger = df.filter((F.col('Body Style') == '2 door') & (F.col('Vehicle Safety Discount') == 0) & (F.col('Antique Vehicle') == 1) & (F.col('Claim Payout') != 0))
print("The count of cars that are dangerous and have Body Style = 2 door and Vehicle Safety Discount = 0: " + str(df_car_danger.select("CAR_ID").distinct().count()))
print("****************************\n")
print("The list of CAR_ID that are dangerous in the order of ascending CAR_ID :")
df_car_danger.select("CAR_ID").distinct().orderBy("CAR_ID").show()



 Assumption: The cars that have [Vehicle Safety Discount] = 0, [Body Style] = 2 door, [Antique Vehicle] =1 and Claim Payout > 0 are the dangerous
****************************

The count of cars that are dangerous and have Body Style = 2 door and Vehicle Safety Discount = 0: 21
****************************

The list of CAR_ID that are dangerous in the order of ascending CAR_ID :
+------+
|CAR_ID|
+------+
|  5334|
| 15383|
| 20957|
| 71463|
| 80176|
| 90363|
|294272|
|334273|
|340589|
|409811|
|426680|
|435664|
|468386|
|576258|
|605396|
|690745|
|730266|
|751772|
|946794|
|947315|
+------+
only showing top 20 rows



 6. How did you define “safe” versus “dangerous”?

Reference: https://www.iihs.org/topics/vehicle-size-and-weight

 Defining Safe cars are the those that have Vehicle Safety Discount = 1, Body Style = Truck and Antique Vehicle = 0 because out of
 all the bod type Truck are considered as safest in a collision as per the reference.
 And Antique cars are old cars and will not be as safe as non-Antique cars.
 Car insurance claim payouts is the financial coverage to deal with any repairs and injuries that may have resulted from the accident.
 So if the claim payout == 0 then we can consider it safer than those cars which has claim payout > 0 which evetuallu means
 the car has been in an accident.

 Defining Dangerous cars are the those that have Vehicle Safety Discount = 0, Body Style = 2 and Antique Vehicle = 1 door because out of
 the body type 2 door are considered as dangerous in a collision as per reference as '2 door' body stype they are the smallest car stype
 out of all options avaliable inder body stype.
 Antique cars can be dangerous cars because they are not meant to be used as day to day use but ocassionally.
 Claim Payout > 0 can be will be more dangerous than the cars that have claim payout amount = 0 as claim payout is the
 financial coverage to deal with repairs and injuries resulted from accident

In [202]:
# 7. Which states have the largest households?

df_State = df.select("State", "HH_ID", "Active HH")
df_State = df_State.withColumnRenamed("Active HH", "ActiveHH")
df_State = df_State.filter("ActiveHH == '1'")
df_State = df_State.distinct()
df_State = (df_State.groupBy("State").agg(countDistinct("HH_ID"))).orderBy(df.State)
df_State = df_State.orderBy(desc("count(DISTINCT HH_ID)"))
df_State.agg({"count(DISTINCT HH_ID)": "sum"})
#df_State.show()
print("While determining the states that have largest households, there are 10 households that have Active HH flag = 1 for 2 different \n\
states so the total count of households are 10 more than actual distinct count of households. ")

print("****************************\n")
df_State.show()
print("****************************\n")
print("State with the largest households: ")
df_State.first()





While determining the states that have largest households, there are 10 households that have Active HH flag = 1 for 2 different 
states so the total count of households are 10 more than actual distinct count of households. 
****************************

+-----+---------------------+
|State|count(DISTINCT HH_ID)|
+-----+---------------------+
|   KY|                 2227|
|   SC|                 2207|
|   AK|                 2197|
|   MN|                 2187|
|   CT|                 2183|
|   WY|                 2181|
|   SD|                 2173|
|   AL|                 2161|
|   MA|                 2155|
|   NV|                 2150|
|   AR|                 2150|
|   WA|                 2149|
|   NM|                 2149|
|   NY|                 2147|
|   MI|                 2139|
|   GA|                 2137|
|   OR|                 2131|
|   HI|                 2131|
|   MO|                 2125|
|   NE|                 2124|
+-----+---------------------+
only showing top 20 rows



Row(State='KY', count(DISTINCT HH_ID)=2227)

In [220]:
# 8. What is the average age of customers?

df_age = df.filter(df.DOB.isNotNull())
df_age = df_age.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))
df_age = df_age.agg({'Age': 'avg'})
print("The average age of customers is :")
df_age.show()
print("##################################")
print("SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG")
print("##################################\n")
df_age_Active = df_withActive.filter(df.DOB.isNotNull())
df_age_Active = df_age_Active.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))
df_age_Active = df_age_Active.agg({'Age': 'avg'})
print("The average age of customers is :")
df_age_Active.show()

The average age of customers is :
+--------------+
|      avg(Age)|
+--------------+
|49.99951999904|
+--------------+

##################################
SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG
##################################

The average age of customers is :
+-----------------+
|         avg(Age)|
+-----------------+
|49.98940168895004|
+-----------------+



In [221]:
# 9. How much does age vary by region?


# creating a region column
df_region = df.withColumn("Region", expr("CASE WHEN State IN ('CT', 'ME', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT') THEN 'Northeast' \
                                               WHEN State IN ('IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI') THEN 'Midwest' \
                                               WHEN State IN ('AL', 'AR', 'DE', 'DC', 'FL', 'GA', 'KY', 'LA', 'MD', 'MS', 'NC', 'OK', 'SC', 'TN', 'TX', 'VA', 'WV') THEN 'South' \
                                               WHEN State IN ('AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY') THEN 'West' \
                                               ELSE 'other' END"))

df_region = df_region.filter(df_region.DOB.isNotNull())
df_region = df_region.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))


df_region.select("Age", "Region").orderBy("Region", "Age")
df_region_min = df_region.groupBy("Region").agg({'age': 'min'})
df_region_max = df_region.groupBy("Region").agg({'age': 'max'})
df_region_max = df_region_max.withColumnRenamed("Region", "Region_max")

df_region = df_region_min.join(df_region_max, df_region_min.Region == df_region_max.Region_max)
df_region = df_region.drop("Region_max")
df_region.show()
print("##################################")
print("SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG")
print("##################################\n")
df_region_Active = df_withActive.withColumn("Region", expr("CASE WHEN State IN ('CT', 'ME', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT') THEN 'Northeast' \
                                               WHEN State IN ('IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI') THEN 'Midwest' \
                                               WHEN State IN ('AL', 'AR', 'DE', 'DC', 'FL', 'GA', 'KY', 'LA', 'MD', 'MS', 'NC', 'OK', 'SC', 'TN', 'TX', 'VA', 'WV') THEN 'South' \
                                               WHEN State IN ('AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY') THEN 'West' \
                                               ELSE 'other' END"))

df_region_Active = df_region_Active.filter(df_region_Active.DOB.isNotNull())
df_region_Active = df_region_Active.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))


df_region_Active.select("Age", "Region").orderBy("Region", "Age")
df_region_min_A = df_region_Active.groupBy("Region").agg({'age': 'min'})
df_region_max_A = df_region_Active.groupBy("Region").agg({'age': 'max'})
df_region_max_A = df_region_max_A.withColumnRenamed("Region", "Region_max")

df_region_Active = df_region_min_A.join(df_region_max_A, df_region_min_A.Region == df_region_max_A.Region_max)
df_region_Active = df_region_Active.drop("Region_max")
df_region_Active.show()




+---------+--------+--------+
|   Region|min(age)|max(age)|
+---------+--------+--------+
|  Midwest|      15|     100|
|    South|      15|     100|
|     West|      15|     100|
|Northeast|      15|     100|
+---------+--------+--------+

##################################
SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG
##################################

+---------+--------+--------+
|   Region|min(age)|max(age)|
+---------+--------+--------+
|  Midwest|      15|     100|
|    South|      15|     100|
|     West|      15|     100|
|Northeast|      15|     100|
+---------+--------+--------+



In [233]:
# 10. Which age group has the most expensive claims?
df_age_group = df.filter(df.DOB.isNotNull())
df_age_group = df_age_group.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))
# creating a column with age_group
df_age_group = df_age_group.withColumn("Age_Group", expr("CASE WHEN Age > 14 AND Age<=20 THEN '15-20' \
                                                                WHEN Age > 20 AND Age<=30 THEN '21-30' \
                                                                WHEN Age > 30 AND Age<=40 THEN '31-40' \
                                                                WHEN Age > 40 AND Age<=50 THEN '41-50' \
                                                                WHEN Age > 50 AND Age<=60 THEN '51-60' \
                                                                WHEN Age > 60 AND Age<=70 THEN '61-70' \
                                                                WHEN Age > 70 AND Age<=80 THEN '71-80' \
                                                                WHEN Age > 80 AND Age<=90 THEN '81-90' \
                                                                WHEN Age > 90 AND Age<=100 THEN '91-100' \
                                                                ELSE 'other' END"))

df_age_group = df_age_group.select("Claim Payout", "Age_Group")
df_age_group = df_age_group.groupBy("Age_Group").agg({'Claim Payout': 'sum'})
df_age_group = df_age_group.orderBy(desc("sum(Claim Payout)"))
df_age_group.show()

print("The Age Group with the most expensive claims: "+str(df_age_group.first()))

print("\n##################################")
print("SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG")
print("##################################\n")
df_age_group_A = df_withActive.filter(df.DOB.isNotNull())
df_age_group_A = df_age_group_A.withColumn('Age', (F.months_between(F.current_date(), F.col('DOB')) / 12).cast('int'))
# creating a column with age_group
df_age_group_A = df_age_group_A.withColumn("Age_Group", expr("CASE WHEN Age > 14 AND Age<=20 THEN '15-20' \
                                                                WHEN Age > 20 AND Age<=30 THEN '21-30' \
                                                                WHEN Age > 30 AND Age<=40 THEN '31-40' \
                                                                WHEN Age > 40 AND Age<=50 THEN '41-50' \
                                                                WHEN Age > 50 AND Age<=60 THEN '51-60' \
                                                                WHEN Age > 60 AND Age<=70 THEN '61-70' \
                                                                WHEN Age > 70 AND Age<=80 THEN '71-80' \
                                                                WHEN Age > 80 AND Age<=90 THEN '81-90' \
                                                                WHEN Age > 90 AND Age<=100 THEN '91-100' \
                                                                ELSE 'other' END"))

df_age_group_A = df_age_group_A.select("Claim Payout", "Age_Group")
df_age_group_A = df_age_group_A.groupBy("Age_Group").agg({'Claim Payout': 'sum'})
df_age_group_A = df_age_group_A.orderBy(desc("sum(Claim Payout)"))
df_age_group_A.show()

print("The Age Group with the most expensive claims with HH Active Flag: "+str(df_age_group_A.first()))




+---------+-----------------+
|Age_Group|sum(Claim Payout)|
+---------+-----------------+
|    21-30|        118133477|
|    31-40|         90089373|
|    15-20|         66604353|
|    41-50|         62508267|
|    71-80|         54957250|
|    61-70|         54059108|
|   91-100|         53632532|
|    51-60|         53449706|
|    81-90|         53061115|
+---------+-----------------+

The Age Group with the most expensive claims: Row(Age_Group='21-30', sum(Claim Payout)=118133477)

##################################
SAME RESULTS BUT CALCULATED FOR THE COMBINED DATASET WITH RECORDS OF HH ACTIVE FLAG
##################################

+---------+-----------------+
|Age_Group|sum(Claim Payout)|
+---------+-----------------+
|    21-30|         97490451|
|    31-40|         72440214|
|    15-20|         52985324|
|    41-50|         51815669|
|    51-60|         45539352|
|    61-70|         45018533|
|   91-100|         43440467|
|    71-80|         42710658|
|    81-90|         41218